![Shopping trolley in front of a laptop](./iStock-1249219777.jpg)

It's simple to buy any product with a click and have it delivered to your door. Online shopping has been rapidly evolving over the last few years, making our lives easier. But behind the scenes, e-commerce companies face a complex challenge that needs to be addressed. 

Uncertainty plays a big role in how the supply chains plan and organize their operations to ensure that the products are delivered on time. These uncertainties can lead to challenges such as stockouts, delayed deliveries, and increased operational costs.

You work for the Sales & Operations Planning (S&OP) team at a multinational e-commerce company. They need your help to assist in planning for the upcoming end-of-the-year sales. They want to use your insights to plan for promotional opportunities and manage their inventory. This effort is to ensure they have the right products in stock when needed and ensure their customers are satisfied with the prompt delivery to their doorstep.


## The Data

You are provided with a sales dataset to use. A summary and preview are provided below.

# Online Retail.csv

| Column     | Description              |
|------------|--------------------------|
| `'InvoiceNo'` | A 6-digit number uniquely assigned to each transaction |
| `'StockCode'` | A 5-digit number uniquely assigned to each distinct product |
| `'Description'` | The product name |
| `'Quantity'` | The quantity of each product (item) per transaction |
| `'UnitPrice'` | Product price per unit |
| `'CustomerID'` | A 5-digit number uniquely assigned to each customer |
| `'Country'` | The name of the country where each customer resides |
| `'InvoiceDate'` | The day and time when each transaction was generated `"MM/DD/YYYY"` |
| `'Year'` | The year when each transaction was generated |
| `'Month'` | The month when each transaction was generated |
| `'Week'` | The week when each transaction was generated (`1`-`52`) |
| `'Day'` | The day of the month when each transaction was generated (`1`-`31`) |
| `'DayOfWeek'` | The day of the weeke when each transaction was generated <br>(`0` = Monday, `6` = Sunday) |

In [3]:
# Import required libraries
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.regression import RandomForestRegressor
from pyspark.sql.functions import col, dayofmonth, month, year,  to_date, to_timestamp, weekofyear, dayofweek
from pyspark.ml.feature import StringIndexer
from pyspark.ml.evaluation import RegressionEvaluator

# Initialize Spark session
my_spark = SparkSession.builder.appName("SalesForecast").getOrCreate()

# Importing sales data
sales_data = my_spark.read.csv(
    "Online Retail.csv", header=True, inferSchema=True, sep=",")

# Convert InvoiceDate to datetime 
sales_data = sales_data.withColumn("InvoiceDate", to_date(
    to_timestamp(col("InvoiceDate"), "d/M/yyyy H:mm")))

In [4]:
# Insert the code necessary to solve the assigned problems. Use as many code cells as you need.
from pyspark.sql.functions import lit, sum as spark_sum

# ── 1. Aggregate daily quantity per Country & StockCode ───────
daily_data = sales_data.groupBy("Country", "StockCode", "InvoiceDate") \
    .agg(spark_sum("Quantity").alias("Quantity"),
         spark_sum("Quantity").alias("Quantity_sum"))

# Keep required columns
daily_data = daily_data.select("Country", "StockCode", "InvoiceDate", "Quantity")

# ── 2. Add time features ──────────────────────────────────────
daily_data = daily_data \
    .withColumn("Year",      year(col("InvoiceDate"))) \
    .withColumn("Month",     month(col("InvoiceDate"))) \
    .withColumn("Week",      weekofyear(col("InvoiceDate"))) \
    .withColumn("Day",       dayofmonth(col("InvoiceDate"))) \
    .withColumn("DayOfWeek", dayofweek(col("InvoiceDate")))

# ── 3. Train/Test split on "2011-09-25" ───────────────────────
split_date = "2011-09-25"

train_data = daily_data.filter(col("InvoiceDate") <= lit(split_date))
test_data  = daily_data.filter(col("InvoiceDate") >  lit(split_date))

# Convert train to pandas as required
pd_daily_train_data = train_data.toPandas()
print("Train shape:", pd_daily_train_data.shape)
print(pd_daily_train_data[["Country", "StockCode", "InvoiceDate", "Quantity"]].head())

# ── 4. Encode categorical features ───────────────────────────
country_indexer   = StringIndexer(inputCol="Country",   outputCol="CountryIndex",   handleInvalid="keep")
stockcode_indexer = StringIndexer(inputCol="StockCode",  outputCol="StockCodeIndex", handleInvalid="keep")

# ── 5. Assemble features ──────────────────────────────────────
feature_cols = ["CountryIndex", "StockCodeIndex", "Year", "Month", "Week", "Day", "DayOfWeek"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# ── 6. Random Forest Regressor ────────────────────────────────
rf = RandomForestRegressor(featuresCol="features", labelCol="Quantity",
                           numTrees=100, maxBins=4000, seed=42)

# ── 7. Pipeline ───────────────────────────────────────────────
pipeline = Pipeline(stages=[country_indexer, stockcode_indexer, assembler, rf])

# ── 8. Train ──────────────────────────────────────────────────
pipeline_model = pipeline.fit(train_data)

# ── 9. Evaluate on test set → MAE ────────────────────────────
predictions = pipeline_model.transform(test_data)

evaluator = RegressionEvaluator(labelCol="Quantity", predictionCol="prediction", metricName="mae")
mae = evaluator.evaluate(predictions)
print(f"\nMAE: {mae:.4f}")

# ── 10. Quantity sold in Week 39 of 2011 ─────────────────────
w39_data = daily_data.filter((col("Week") == 39) & (col("Year") == 2011))
w39_preds = pipeline_model.transform(w39_data)

quantity_sold_w39 = int(w39_preds.agg(spark_sum("prediction").alias("total")).collect()[0]["total"])
print(f"\nEstimated quantity sold in Week 39, 2011: {quantity_sold_w39}")

Train shape: (175452, 9)
          Country StockCode InvoiceDate  Quantity
0  United Kingdom     22537  2010-01-12        24
1  United Kingdom     22716  2010-01-12        12
2  United Kingdom     22953  2010-02-12         1
3  United Kingdom     22424  2010-02-12         5
4          France     21791  2010-03-12        12


26/04/30 08:17:23 WARN DAGScheduler: Broadcasting large task binary with size 1936.0 KiB


26/04/30 08:17:25 WARN DAGScheduler: Broadcasting large task binary with size 4.0 MiB


26/04/30 08:17:31 WARN DAGScheduler: Broadcasting large task binary with size 7.9 MiB


26/04/30 08:17:40 WARN DAGScheduler: Broadcasting large task binary with size 12.8 MiB



MAE: 9.3625

Estimated quantity sold in Week 39, 2011: 88223
